## XBRL US API - Search SEC report extension concepts for keyword strings  
### Authenticate for access token 
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# @title
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode


class tokenInfoClass:
    access_token = None
    refresh_token = None
    email = None
    username = None
    client_id = None
    client_secret = None
    url = 'https://api.xbrl.us/oauth2/token'
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
	
def refresh(info):
    refresh_auth = {
                'client_id': info.client_id, 
				'client_secret' : info.client_secret, 
				'grant_type' : 'refresh_token', 
				'platform' : 'ipynb', 
				'refresh_token' : info.refresh_token 
                }
    refreshres = requests.post(info.url, data=refresh_auth, headers=info.headers)
    refresh_json = refreshres.json()
    info.access_token = refresh_json['access_token']
    info.refresh_token = refresh_json['refresh_token']
    print('Your access token(%s) is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.' % (info.access_token))
    return info	

tokenInfo = tokenInfoClass()

tokenInfo.email = input('Enter your XBRL US Web account email: ')
tokenInfo.password = getpass.getpass(prompt='Password: ')
tokenInfo.client_id = getpass.getpass(prompt='Client ID: ')
tokenInfo.client_secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : tokenInfo.email, 
            'client_id': tokenInfo.client_id, 
            'client_secret' : tokenInfo.client_secret, 
            'password' : tokenInfo.password, 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

#print(body_auth)

payload = urlencode(body_auth)
res = requests.request("POST", tokenInfo.url, data=payload, headers=tokenInfo.headers)
auth_json = res.json()

if 'error' in auth_json:
    print("\n\nThere was a problem generating the access token: %s.  Run the first cell again and enter the credentials." % (auth_json['error_description']))
else:
    tokenInfo.access_token = auth_json['access_token']
    tokenInfo.refresh_token = auth_json['refresh_token']
    print ("\n\nYour access token expires in 60 minutes. After it expires, it should be regenerated automatically.  If not, run the cell rerun the first query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
	
#print(vars(tokenInfo))
print('\n\naccess token: ' + tokenInfo.access_token + ' refresh token: ' + tokenInfo.refresh_token)

## Define search filters and fields to return

The section below sets the fact endpoint, keyword strings to evaluate extended fact concepts, and report attributes for the search, along  with fact attributes to be returned for each matching fact.

In [2]:
### Define the parameters for the filter and fields to be returned

# Define endpoint (common values: fact, entity, report, cube, label, concept, relationship - see https://xbrlus/github.io/xbrl-api for additional endpoint options)

endpoint = 'fact'

Keyword_List = [
    'government',
    'governmentassistance',
    'governmentgrant',
    'governmentincentive',
    'taxincentive'
                ]

report_types = ['10-K', '10-K/A'
                ]

years = ['2020','2021','2022','2023','2024',] ## Use commas between for multiple years, e.g., '2018','2019'
#years = [str(2013 + i) for i in range(8)] ## Years 2013 to 2020

fields = [
         'dimensions.count.sort(ASC)',
         'entity.name.sort(ASC)',
         'report.sic-code',
         'dts.id',
         #'fact.id',
         'report.filing-date',
         'period.fiscal-year',
         'period.instant',
         'report.document-type',
         'concept.local-name.sort(ASC)',
         'concept.is-base',
         'dimension-pair',
         'fact.value',
         'fact.decimals',
         'dimension.namespace',
         'member.namespace'
         ]

# Set unique rows as True of False (True drops any duplicate rows)
unique = True

# Limit the number of rows displayed by the notebook (does not impact the data frame)
rows_to_display = 20 # Set as '' to display all rows in the notebook

params = {
     'report.source-name': 'SEC',
     'concept.is-base': 'FALSE',
     'concept.is-monetary': 'TRUE',
     'fact.accuracy-index': '1',
     'period.fiscal-period': 'Y',
     'period.fiscal-year': ','.join(years),
     'report.document-type': ','.join(report_types),
     'fields': ','.join(fields)
     }

print('click the run button below to execute this query')

click the run button below to execute this query


## Query and results

The code below iterates for all facts matching the defined query parameters against all SICs used in reports between 2020 and 2024.  At the conclusion of the process, the dataframe is filtered by the keyword list, so the displayed results show only extension concepts containing the keyword strings.

In [ ]:
# @title
sic_code = [ # used in recent 10/K reports (login to view in browser): https://api.xbrl.us/api/v1/report/search?report.document-type=10-K,10-K/A&unique&report.year-focus=2024,2023,2022,2021,2020&fields=report.sic-code.sort(ASC)
          "3330","3334","3341","3350","3357","3360","3390","3411","3412","3420",
          "3430","3433","3440","3442","3443","3448","3460","3470","3480","3490",
          "3510","3523","3524","3530","3531","3533","3537","3540","3541","3550",
          "3559","3560","3561","3562","3564","3567","3569","3570","3571","3572",
          "3576","3577","3578","3579","3580","3585","3590","3600","3612","3613",
          "7819","7822","7830","7841","7900","7948","7990","7997","8000","8011",
          "8050","8051","8060","8062","8071","8082","8090","8093","8111","8200",
          "8351","8700","8711","8731","8734","8741","8742","8744","8900"
          "100","200","700","900","1000","1040","1090","1220","1221","1311",
          "1381","1382","1389","1400","1520","1531","1540","1600","1623","1700",
          "1731","2000","2011","2013","2015","2020","2024","2030","2033","2040",
          "2050","2052","2060","2070","2080","2082","2086","2090","2092","2100",
          "2111","2200","2211","2221","2273","2300","2320","2330","2340","2390",
          "2400","2421","2430","2451","2510","2511","2520","2522","2531","2600",
          "2611","2621","2631","2650","2670","2673","2711","2721","2731","2750",
          "2761","2780","2800","2810","2820","2821","2833","2834","2835","2836",
          "2840","2842","2844","2851","2860","2870","2890","2891","2911","2990",
          "3011","3021","3050","3060","3080","3081","3086","3089","3100","3140",
          "3211","3221","3231","3241","3260","3272","3290","3310","3312","3317",
          "3620","3621","3630","3634","3640","3651","3652","3661","3663","3669",
          "3670","3672","3674","3677","3678","3679","3690","3711","3713","3714",
          "3715","3716","3720","3721","3724","3728","3730","3743","3751","3760",
          "3790","3812","3821","3822","3823","3824","3825","3826","3827","3829",
          "3841","3842","3843","3844","3845","3851","3861","3873","3910","3942",
          "3944","3949","3990","4011","4013","4100","4210","4213","4220","4400",
          "4412","4512","4513","4522","4581","4610","4700","4731","4812","4813",
          "4822","4832","4833","4841","4899","4900","4911","4922","4923","4924",
          "4931","4932","4941","4950","4953","4955","4991","5000","5010","5013",
          "5020","5030","5031","5040","5045","5047","5050","5051","5063","5065",
          "5070","5072","5080","5084","5090","5094","5099","5110","5122","5130",
          "5140","5141","5150","5160","5171","5172","5180","5190","5200","5211",
          "5311","5331","5400","5411","5412","5500","5531","5600","5621","5651",
          "5661","5700","5712","5731","5734","5735","5810","5812","5900","5912",
          "5940","5944","5945","5960","5961","5990","6021","6022","6029","6035",
          "6036","6099","6111","6141","6153","6159","6162","6163","6189","6199",
          "6200","6211","6221","6282","6311","6321","6324","6331","6351","6361",
          "6399","6411","6500","6510","6512","6513","6519","6531","6552","6770",
          "6792","6794","6795","6798","6799","7000","7011","7200","7310","7311",
          "7320","7330","7331","7340","7350","7359","7361","7363","7370","7371",
          "7372","7373","7374","7380","7381","7389","7500","7510","7600","7812",
            ]
### Execute the query with loop for all results 
### THIS SECTION DOES NOT NEED TO BE EDITED

search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
if unique:
    search_endpoint += "?unique"
orig_fields = params['fields']
res_df = []
query_start = datetime.now()
                
import math
total_sics = len(sic_code)
sic_batch_num = math.floor(total_sics / 5)
rounds = math.ceil(total_sics/sic_batch_num)
round_num = 1
total_rows = 0
your_limit = 0

for x in range(0, total_sics, sic_batch_num):
    offset_value = 0
    count = 0
    offset_value = 0
    printed = False
    run_query = True
    segment_query_start = datetime.now()
    params['report.sic-code'] = ','.join(sic_code[x:x+sic_batch_num])
    params['fields'] = orig_fields
    print("Round %d/%d\nSic Codes: %s" % (round_num, rounds, ','.join(sic_code[x:x+sic_batch_num])))    
    res_df_segment = []
    #print(params)
               
    while True:
        if not printed:
            print("On", query_start.strftime("%c"), tokenInfo.email, "(client ID:", str(tokenInfo.client_id.split('-')[0]), "...) started the query \n")
            printed = True
        retry = 0
        while retry < 3:
            res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(tokenInfo.access_token)})
            res_json = res.json()
            if 'error' in res_json:
                if res_json['error_description'] == 'Bad or expired token':
                    tokenInfo = refresh(tokenInfo)
                else: 
                    print('There was an error: {}'.format(res_json['error_description']))
                    run_query = False
                    break
            else: 
                    break
            retry +=1
            if retry >= 3:
                print("Can't refresh the access token.  Run the first query block, then rerun the query.")
                run_query = False

        if not run_query:
            break

        print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

        res_df_segment += res_json['data']
        your_limit = res_json['paging']['limit']

        if res_json['paging']['count'] < res_json['paging']['limit']:
            print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
            break
        else: 
            offset_value += res_json['paging']['limit'] 
            if 100 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 10 * res_json['paging']['limit']:
                            break 
            elif 500 == res_json['paging']['limit']:
                    params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                    if offset_value == 4 * res_json['paging']['limit']:
                            break 
            params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
        

    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - segment_query_start
    print("\nAt %s the query \n%s\nfinished with  %d rows returned in %s.\n" % (current_datetime.strftime("%c"), urllib.parse.unquote(res.url), len(res_df_segment), str(time_taken)))
    total_rows += len(res_df_segment)
    round_num += 1
    res_df += res_df_segment
    #if round_num >= 3:
    #    break

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    #index = pd.DataFrame(res_df).index
    #total_rows = len(index)
    #your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    #print("\nAt " + current_datetime.strftime("%c") +  ", the query \n" + urllib.parse.unquote(res.url) + "\nfinished with  ", str(total_rows), "  rows returned in " + str(time_taken) + ".\n")
    #print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with a total of ", str(total_rows), "  rows returned in " + str(time_taken) + ".\n")

    print("\nAt %s, the query finished with a total of %d : %d rows returned in %s.\n" % (current_datetime.strftime("%c"), total_rows, len(res_df), str(time_taken)))
    
    df = pd.DataFrame(res_df)
    filtered_df = df[df['concept.local-name'].str.contains('|'.join(Keyword_List), case=False, na=False)]
    filteredindex = pd.DataFrame(filtered_df).index
    filtered_rows = len(filteredindex)
    print("There are  ", str(filtered_rows), "  extension concepts matching the keyword strings.")
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(filtered_df.to_html(max_rows=rows_to_display)))

In [ ]:
# If you run this program locally, you can save the output to a file 
# on your computer (modify D:\results.csv to your system)

filtered_df.to_csv(r"C:\results.csv",sep=",")

# Google Colab users - comment out the line above and uncomment the code below to save the data frame as a .csv in your Google Drive

#from google.colab import drive
#drive.mount('drive')
#df.to_csv('data.csv')
#!cp data.csv "drive/My Drive/"

## Show results

In [ ]:
columns_to_hide = ['entity.cik', 'fact.id', 'fact.decimals', 'dimension.namespace', 'member.namespace']
columns_to_show = [column for column in filtered_df.columns if column not in columns_to_hide]

In [ ]:
filtered_df.sort_values(by=['entity.name','dts.id','concept.local-name','report.sic-code'], inplace=True)
filtered_df[columns_to_show].head(20)

## Show dimensions example (if exists)

In [ ]:
filtered_df[filtered_df['dimensions.count'] > 1].sort_values(by=['entity.name','dts.id','concept.local-name','dimension.namespace'])[columns_to_show].head(20)